# 01. Dataset Analysis and Exploration

This notebook performs Exploratory Data Analysis (EDA) on the English-Amharic parallel sentence dataset.

### Dataset Reference
- **Source**: [`michsethowusu/english-amharic_sentence-pairs_mt560`](https://huggingface.co/datasets/michsethowusu/english-amharic_sentence-pairs_mt560)
- **Language Pair**: English (`eng`) to Amharic (`amh`)
- **Total Sentence Pairs**: 669,145

### Objectives
1. Inspect columns, missing values, and data shapes.
2. Analyze duplicate sentences and duplicate translation pairs.
3. Compute sequence length distributions and percentiles.
4. Evaluate sequence length thresholds to inform the maximum sequence length (`MAX_LEN = 70`).


In [ ]:
# Environment & Path Setup
# If running on Google Colab, uncomment the lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/english-amharic-nmt

import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import get_data_paths
from src.utils.seed import set_seed

# Configure data directory (can be overridden by DATA_ROOT environment variable)
# On Colab: DATA_ROOT = "/content/drive/MyDrive/english-amharic-nmt-data"
DATA_ROOT = os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))
paths = get_data_paths(DATA_ROOT)
paths.ensure_directories()
set_seed(42)

print("Project root:", PROJECT_ROOT)
print("Data directory:", paths.data_root)


## 1. Load the Parallel Dataset
We load the parallel sentence dataset from Hugging Face Hub (or from local cache if already downloaded).

In [ ]:
from src.data.dataset import load_raw_dataset_from_hf

# Load dataset into pandas DataFrame
df = load_raw_dataset_from_hf(cache_dir=paths.raw_dir)
print(f"Total sentence pairs loaded: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")


## 2. Inspect Sample Sentence Pairs

In [ ]:
for i in range(5):
    print(f"[{i+1}]")
    print("  EN:", df.iloc[i]["eng"])
    print("  AM:", df.iloc[i]["amh"])
    print("-" * 70)


## 3. Missing Values & Duplicate Analysis
We check for null entries, duplicate sentences in each language, and exact duplicate pairs (`eng` + `amh`).

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Check duplicates
duplicate_eng = df["eng"].duplicated().sum()
duplicate_amh = df["amh"].duplicated().sum()
duplicate_pairs = df.duplicated(subset=["eng", "amh"], keep=False)
unique_duplicate_pairs = df[duplicate_pairs][["eng", "amh"]].drop_duplicates().shape[0]

print(f"\nDuplicate English sentences: {duplicate_eng:,}")
print(f"Duplicate Amharic sentences: {duplicate_amh:,}")
print(f"Rows belonging to duplicate pairs: {duplicate_pairs.sum():,}")
print(f"Unique duplicate pairs: {unique_duplicate_pairs:,}")


## 4. Sentence Length Statistics
We compute token lengths using whitespace separation and analyze summary statistics and quantiles.

In [ ]:
df["eng_words"] = df["eng"].astype(str).str.split().str.len()
df["amh_words"] = df["amh"].astype(str).str.split().str.len()

print("English word count statistics:")
print(df["eng_words"].describe())

print("\nAmharic word count statistics:")
print(df["amh_words"].describe())


## 5. Length Percentiles and Truncation Thresholds
To select an optimal `MAX_LEN`, we calculate percentiles from 90% up to 99.9% and inspect sentences exceeding various thresholds.

In [ ]:
print("Length Percentiles:")
percentiles = [0.90, 0.95, 0.99, 0.995, 0.999]
print("English:")
print(df["eng_words"].quantile(percentiles))

print("\nAmharic:")
print(df["amh_words"].quantile(percentiles))

print("\nSentences exceeding length thresholds:")
thresholds = [30, 40, 50, 60, 70, 80, 100]
for t in thresholds:
    eng_count = (df["eng_words"] > t).sum()
    amh_count = (df["amh_words"] > t).sum()
    print(f"> {t:3d} words | English: {eng_count:6,d} ({eng_count/len(df)*100:.2f}%) | Amharic: {amh_count:6,d} ({amh_count/len(df)*100:.2f}%)")


## 6. Conclusions & Preprocessing Decisions
1. **Cleaning**: Exactly 51 duplicate sentence pairs should be removed, leaving 669,094 clean pairs.
2. **Whitespace Normalization**: Irregular whitespace and tabs must be collapsed into single spaces.
3. **Sequence Length**: With `MAX_LEN = 70` (including `<SOS>` and `<EOS>`), over **99.4%** of sentences in both English and Amharic are preserved without truncation, providing an ideal trade-off between computational efficiency and semantic completeness.
